# Creating LLM

In [18]:
#reading the verdict from verdict.txt file
with open('verdict.txt', 'r') as file:
    verdict = file.read().strip()
print("The length of the verdict is:", len(verdict))
print(verdict[:100])

The length of the verdict is: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [19]:
# Using tiktoken library to tokenize the verdict (BPE tokenization)
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
# print(tokenizer.max_token_value+1)
encoded = tokenizer.encode(verdict)
print("The number of tokens in the verdict is:", len(encoded))
print(encoded[:10])

#decoding the token ids back to text
decoded_ids = tokenizer.decode(encoded)
print(decoded_ids[:100])

The number of tokens in the verdict is: 5145
[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138]
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [20]:
# Demo Implementing Data Sampling
context_size = 4
x = encoded[:context_size]
y = encoded[1:context_size+1]
print("X",x)
print("Y",y)

X [40, 367, 2885, 1464]
Y [367, 2885, 1464, 1807]


In [21]:
for i in range(1,context_size+1,2):
    context = encoded[:i]
    target = encoded[:i+1]
    print(f"{tokenizer.decode(context)} ----> {tokenizer.decode(target)}")

I ----> I H
I HAD ----> I HAD always


In [22]:
import torch 
from torch import nn
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, encoded_data, tokenizer, max_length, stride):
        self.input_id = []
        self.target_id = []

        for i in range(0, len(encoded_data)-max_length, stride):
            input_chunk = encoded_data[i:i+max_length]
            target_chunk = encoded_data[i+1:i+max_length+1]
            self.input_id.append(torch.tensor(input_chunk))
            self.target_id.append(torch.tensor(target_chunk))
            
    def __len__(self):
        return len(self.input_id)
    
    def __getitem__(self, idx):
        return self.input_id[idx], self.target_id[idx]

In [23]:
# Demo Implementing Data Sampling
dataset = GPTDataset(encoded, tokenizer, max_length=256, stride=128)
print(dataset)

In [24]:
def create_dataloader(txt,batch_size=8, max_length=4, stride=4,shuffle=False,drop_last = True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    data = GPTDataset(tokenizer.encode(txt), tokenizer, max_length, stride)
    dataLoader = DataLoader(data, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataLoader

In [25]:
dataLoader = create_dataloader(verdict,batch_size=8,max_length=4,stride=4,shuffle=False)
data_itr = iter(dataLoader)
inputs, targets = next(data_itr)
print(inputs)
print(inputs.shape,targets.shape)
print(len(dataLoader))

#Output :
# tensor([[   40,   367,  2885,  1464],
#         [ 1807,  3619,   402,   271],
#         [10899,  2138,   257,  7026],
#         [15632,   438,  2016,   257],
#         [  922,  5891,  1576,   438],
#         [  568,   340,   373,   645],
#         [ 1049,  5975,   284,   502],
#         [  284,  3285,   326,    11]])
# torch.Size([8, 4]) torch.Size([8, 4])

tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
torch.Size([8, 4]) torch.Size([8, 4])
160


In [26]:
# Embedding the input tokens int the form of 8*4*256
token_encodingLayer = nn.Embedding(50257,256) #Embedding layer does know that Every integer in this tensor is an index. Replace it with the corresponding embedding vector
# Output : torch.Size([8, 4, 256])

In [27]:
#Creating positional encodings for the input tokens
pos_embeddingLayer = nn.Embedding(4,256)
pos_encodings = pos_embeddingLayer(torch.arange(4))
print(pos_encodings.shape,pos_encodings)

torch.Size([4, 256]) tensor([[ 0.2735, -0.3832, -0.2215,  ...,  0.2743, -0.2860, -0.2261],
        [-0.6246,  0.0982, -0.6928,  ..., -0.3122,  0.5411,  0.7830],
        [-0.5692,  0.6468, -0.5872,  ...,  0.9307,  0.1529,  0.3800],
        [-0.8663,  0.1074, -1.2733,  ...,  1.2466, -0.8009, -0.5638]],
       grad_fn=<EmbeddingBackward0>)


In [28]:
# Pocessing each batch
for batch_num, (inputs, targets) in enumerate(dataLoader):
    # print(f"Processing Batch {batch_num+1}")
    token_encoding = token_encodingLayer(inputs) 
    input_embeddings = token_encoding + pos_encodings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


## Implementing Casual Attention Class

In [29]:
class CasualSelfAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=True):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias) #Linear transformation to get the query vector
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)   #Linear transformation to get the query vector
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias) #Linear transformation to get the query vector
        self.dropout = nn.Dropout(dropout)
        # register_buffer is similar to nn.prameter but it is not a lernable parameter, it is a constant that is saved with the model
        self.register_buffer("mask", torch.triu(torch.ones(context_length,context_length),diagonal=1))
    
    def forward(self,x):
        b, num_token, d_in = x.shape
        keys = self.W_key(x)
        query = self.W_query(x)
        value = self.W_value(x)

        attn_score = query @ keys.transpose(1,2)
        attn_score = attn_score.masked_fill(self.mask[:num_token,:num_token] == 1, float("-inf"))
        attn_weights = torch.softmax(attn_score / keys.shape[-1]**0.5, dim=-1)

        attn_weights = self.dropout(attn_weights)
        context_vec = attn_weights @ value
        return context_vec

In [30]:
context_length = 4
ca = CasualSelfAttention(d_in=256, d_out=2, context_length=context_length, dropout=0.1)
context_vec = ca(input_embeddings)
print(context_vec.shape)

print('''
This is one of the batch which has 8 sentences with 4 tokens and each token is represented by
256 dimensional vector. After applying the casual self attention we get the context vector of each token
of shape [1,2]. This is for one token, we have 4 tokens in each sentence and 8 sentences in a batch. 
So the output shape is [8,4,2].
''')

torch.Size([8, 4, 2])

This is one of the batch which has 8 sentences with 4 tokens and each token is represented by
256 dimensional vector. After applying the casual self attention we get the context vector of each token
of shape [1,2]. This is for one token, we have 4 tokens in each sentence and 8 sentences in a batch. 
So the output shape is [8,4,2].



## Wrapper class to implement Multi-head attention 

In [31]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.head = nn.ModuleList([
            CasualSelfAttention(d_in=d_in, d_out=d_out, context_length=context_length, dropout=0.1) for _ in range(num_heads)
        ])
    
    def forward(self,x):
        return torch.cat([head(x) for head in self.head], dim=1)

In [32]:
mhaw = MultiHeadAttentionWrapper(d_in=256, d_out=2, context_length=context_length, dropout=0.1, num_heads=2)
context_v = mhaw(input_embeddings)
print(context_v.shape)
# print(context_v)

torch.Size([8, 8, 2])


Since this multihead attention mechanisum works sequentially therefore the computations requried are more.  
Implementing the multihead attention such that it can compute parallely

In [33]:
class MulticlassAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length,dropout,num_heads,qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads) == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias) #Linear transformation to get the query vector
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)   #Linear transformation to get the query vector
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias) #Linear transformation to get the query vector
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length,context_length),diagonal=1))

    def forward(self,x):
        b, num_token, d_in = x.shape
        keys = self.W_key(x).view(b, num_token, self.num_heads, self.head_dim).transpose(1, 2)
        query = self.W_query(x).view(b, num_token, self.num_heads, self.head_dim).transpose(1, 2)
        value = self.W_value(x).view(b, num_token, self.num_heads, self.head_dim).transpose(1, 2)

        attn_Score = query @ keys.transpose(2,3)
        mask_bool = self.mask[:num_token,:num_token]
        attn_Score = attn_Score.masked_fill(mask_bool == 1, float("-inf"))
        attn_weights = torch.softmax(attn_Score / self.head_dim**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ value
        context_vec = context_vec.contiguous().view(b, num_token, self.d_out)

        context_vec = self.out_proj(context_vec)
        return context_vec

### Implementing Attention Module As Per Smallest GPT-2 Architecture

In [34]:
#implementing attention module as per smallest GPT-2 architecture
# No. of attention heads = 12, input and output dimension =768 and context length = 1024

#Preparing the input for the attention module

#step 1 : Tokeninzing the input text into 1024 of context length and embedding it into 768 dimensional vector
with open('verdict.txt', 'r') as file:
    corpus = file.read().strip()

# Step 2: Creating a DataLoader for the encoded corpus which already divides the corpus into input and target of token length 1024 and converts into batch size of 8
DataLoader = create_dataloader(corpus, batch_size = 8, max_length = 1024, stride = 1024, shuffle = False)
print(len(DataLoader))

# Step 3: Embedding the input tokens into 768 dimensional vector and adding positional encodings
TokenEmbeddingLayer = nn.Embedding(50257,768)
PositionalEmbeddingLayer = nn.Embedding(1024,768)
positional_encodings = PositionalEmbeddingLayer(torch.arange(1024))

for batch_num, (inputs, targets) in enumerate(DataLoader):
    Token_embeddings = TokenEmbeddingLayer(inputs)
    Input_embeddings = Token_embeddings + positional_encodings
# print(Input_embeddings.shape)

# Step 4: Passing the input embeddings through the multi head attention module with 12 heads and output dimension of 768
mha = MulticlassAttention(d_in=768, d_out=768, context_length=1024, dropout=0.1, num_heads=12)
context_vector = mha(Input_embeddings)
print(context_vector.shape)

0


NameError: name 'Input_embeddings' is not defined

# Implementing GPT Model from Scratch

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size" : 50257,
    "context_length" : 1024,
    "emb_dim" : 768,
    "n_heads" : 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [ ]:
# A Placeholder GPT model architecture class

class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.dropout = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_emb = self.tok_emb(in_idx)
        pos_emb = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_emb + pos_emb
        x = self.dropout(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5 # is a small constant added to the varuance to prevent division by zero
        self.scale = nn.Parameter(torch.ones(emb_dim)) # learnable parameter that scales the normalized output
        self.shift = nn.Parameter(torch.zeros(emb_dim)) # learnable parameter that shifts the normalized output
    
    def forward(self,x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * x_norm + self.shift


'''
GELU and SwiGLU ae more complex and smooth activation functions incorporating Gaussaian and sigmoid-gated linear units respectively.
Unlike ReLU which outputs zero for any negative input, GELU allows for small non-negative output for negative values. 
'''

class GELU(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self,x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))

class FeedForward(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self,x):
        return self.layers(x)

#### Creating Transformer Block Component

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MulticlassAttention(d_in=cfg["emb_dim"],
                                       d_out=cfg["emb_dim"],
                                       context_length=cfg["context_length"],
                                       num_heads=cfg["n_heads"],
                                       dropout=cfg["drop_rate"],
                                       qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
    
    def forward(self,x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x

In [ ]:
#testing the transformer block with dummy input
torch.manual_seed(123)
dummy_input = torch.rand(8, 4, 768)
trf_block = TransformerBlock(GPT_CONFIG_124M)
output = trf_block(dummy_input)
print(dummy_input.shape)
print(output.shape)

torch.Size([8, 4, 768])
torch.Size([8, 4, 768])


In [ ]:
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day hold a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch)
print(batch)

model = DummyGPTModel(GPT_CONFIG_124M)
logits = model(batch)
print(logits.shape)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 1745,  257]])
torch.Size([2, 4, 50257])


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters in the model: {total_params:,}")
print(f"Size of the model in GB: {total_params * 4 / (1024**2):.2f} MB")

Total number of parameters in the model: 163,009,536
Size of the model in GB: 621.83 GB


In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probas, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)

model.eval()
out = generate_text_simple(model, idx=encoded_tensor, max_new_tokens=10, context_size=GPT_CONFIG_124M["context_length"])
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

Hello, I amanderHameeawarenessmedauto 320ernandez injection Rush
